# Lab 09 — Diffusion Models from Scratch

**Deep Learning for Computer Vision · AIT**

You are going to build a generative model that starts from pure noise and produces
recognisable images, and you are going to build every piece of it: the noise schedule,
the time-conditioned network, the training objective, two samplers, and
classifier-free guidance.

The whole thing rests on one idea:

> Destroying an image is easy and has a closed form. **Learn to undo one step of the
> destruction**, then run that in reverse many times.

**Runs on:** Colab free T4, ~30 min to train. Sampling is seconds.

**Reading:** [DDPM](https://arxiv.org/abs/2006.11239) ·
[DDIM](https://arxiv.org/abs/2010.02502) ·
[Classifier-free guidance](https://arxiv.org/abs/2207.12598) ·
[Improved DDPM](https://arxiv.org/abs/2102.09672) ·
[Lilian Weng's diffusion post](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/)

## 0. What came before: GANs, in five minutes

For roughly six years (2014–2020) generative vision *meant* GANs. You should know the
idea, because the vocabulary is everywhere and because its failure modes explain why
diffusion won.

A **GAN** is two networks in a game. The **generator** G maps noise z to an image. The
**discriminator** D tries to tell real images from generated ones. They train against
each other:

- D is trained to output 1 on real data and 0 on G's output.
- G is trained to make D output 1 on its output.

At equilibrium G's distribution matches the data and D is reduced to guessing. GANs
produce very sharp images very fast — a single forward pass.

**Why the field moved on:**

| Problem | What it looks like |
|---|---|
| **Mode collapse** | G finds a few images that fool D and produces only those. Your "face generator" makes the same three faces |
| **Unstable training** | The two losses oscillate; there is no single number that says "training is going well" |
| **No likelihood** | You cannot ask "how probable is this image?", which makes evaluation awkward (hence FID) |
| **Hard to condition** | Text-conditioning a GAN at scale proved very difficult |

Wasserstein GAN with gradient penalty fixed much of the instability, but not the
underlying awkwardness. Diffusion models trade GAN's one-shot speed for a **stable
regression objective** — literally mean-squared error — that scales predictably and
conditions easily. That trade turned out to be the right one.

*The full GAN lab from earlier versions of this course is preserved in `archive/GANs/`
if you want the implementation details of DCGAN, conditional GAN and WGAN-GP.*

## 1. Setup

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt, math, time

device = "cuda" if torch.cuda.is_available() else "cpu"
AMP_DTYPE = torch.float16          # T4 = Turing: fp16, not bf16
torch.manual_seed(0); np.random.seed(0)
print("device:", device)

device: cuda


In [3]:
IMG_SIZE, CHANNELS, NUM_CLASSES = 32, 1, 10

tf = T.Compose([
    T.Resize(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.5], [0.5]),         # -> [-1, 1], which is what the noise schedule assumes
])

train_set = torchvision.datasets.FashionMNIST("./data", train=True, download=True, transform=tf)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True,
                                           num_workers=2, pin_memory=True, drop_last=True)
LABELS = train_set.classes
print(len(train_set), "images |", LABELS)

RuntimeError: Error downloading train-images-idx3-ubyte.gz:
Tried http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/, got:
<urlopen error [Errno 111] Connection refused>


---
# 2. The forward process: destroying an image

We add Gaussian noise over T steps. Each step is

$$q(x_t \mid x_{t-1}) = \mathcal{N}\!\left(x_t;\ \sqrt{1-\beta_t}\,x_{t-1},\ \beta_t I\right)$$

where the schedule β₁ … β_T grows slowly from about 1e-4 to 0.02.

The crucial property — the reason any of this is trainable — is that composing many
Gaussians gives another Gaussian **in closed form**. Writing αₜ = 1 − βₜ and
ᾱₜ = ∏ αₛ:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

So we can jump to **any** timestep in one line, with no loop. That is what makes
training cheap: sample a random t per image, jump there, ask the network to identify the
noise.

We use the **cosine schedule** from Improved DDPM rather than the original linear one —
the linear schedule destroys small images like ours too early, wasting a large fraction
of the timesteps on inputs that are already indistinguishable from noise.

In [ ]:
T_STEPS = 400          # small enough to train fast, large enough to look good


def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    ac = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    ac = ac / ac[0]
    betas = 1 - (ac[1:] / ac[:-1])
    return betas.clamp(1e-4, 0.999)


betas = cosine_beta_schedule(T_STEPS).to(device)
alphas = 1.0 - betas
abar = torch.cumprod(alphas, dim=0)                        # alpha-bar
abar_prev = F.pad(abar[:-1], (1, 0), value=1.0)

sqrt_abar = abar.sqrt()
sqrt_one_minus_abar = (1 - abar).sqrt()
posterior_var = betas * (1 - abar_prev) / (1 - abar)       # variance of q(x_{t-1} | x_t, x_0)

plt.figure(figsize=(11, 3.5))
plt.subplot(1, 2, 1); plt.plot(betas.cpu()); plt.title("beta_t (cosine schedule)"); plt.grid(alpha=.3)
plt.subplot(1, 2, 2)
plt.plot(abar.cpu(), label="alpha-bar (signal)")
plt.plot((1 - abar).cpu(), label="1 - alpha-bar (noise)")
plt.title("signal vs noise over time"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
def q_sample(x0, t, noise=None):
    # Jump straight to timestep t. One line, no loop.
    if noise is None:
        noise = torch.randn_like(x0)
    a = sqrt_abar[t].view(-1, 1, 1, 1)
    b = sqrt_one_minus_abar[t].view(-1, 1, 1, 1)
    return a * x0 + b * noise


x0, _ = next(iter(train_loader))
x0 = x0[:1].to(device)
steps = [0, 40, 80, 140, 200, 280, 340, T_STEPS - 1]

fig, ax = plt.subplots(1, len(steps), figsize=(15, 2.6))
for a, s in zip(ax, steps):
    xt = q_sample(x0, torch.tensor([s], device=device))
    a.imshow(xt[0, 0].cpu(), cmap="gray", vmin=-1, vmax=1)
    a.set_title("t = %d" % s, fontsize=10); a.axis("off")
plt.suptitle("Forward process: signal dissolving into noise"); plt.tight_layout(); plt.show()

---
# 3. The model: a U-Net that knows what time it is

The network's job is: given a noisy image x_t **and** the timestep t, predict the noise ε
that was added.

Two additions to the U-Net from Lab 08:

1. **Timestep embedding.** t is turned into a vector with the same sinusoidal encoding
   used for Transformer positions (Lab 06), passed through a small MLP, and **added to the
   feature maps inside every block**. Without this the network cannot know how much noise
   to expect, and one network cannot serve all T steps.
2. **Class embedding**, for the conditional generation in Section 6. A learned embedding
   per class, added to the same conditioning vector. Class index `NUM_CLASSES` is reserved
   for "unconditional" — that is the trick that makes classifier-free guidance possible.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    # Identical construction to Transformer positional encoding, applied to t rather than position.
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        args = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, cin, cout, emb_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, cin)
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, cout)             # conditioning enters here
        self.norm2 = nn.GroupNorm(8, cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]  # broadcast over H, W
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class SelfAttention(nn.Module):
    # One attention block at the low-resolution stage: cheap, and it captures global structure.
    def __init__(self, ch, heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.attn = nn.MultiheadAttention(ch, heads, batch_first=True)

    def forward(self, x):
        b, c, h, w = x.shape
        y = self.norm(x).view(b, c, h * w).transpose(1, 2)
        y, _ = self.attn(y, y, y)
        return x + y.transpose(1, 2).view(b, c, h, w)

In [ ]:
class DiffusionUNet(nn.Module):
    def __init__(self, ch=64, in_ch=CHANNELS, num_classes=NUM_CLASSES):
        super().__init__()
        emb = ch * 4
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(ch), nn.Linear(ch, emb), nn.SiLU(), nn.Linear(emb, emb))
        # +1 class slot = the "unconditional" token used by classifier-free guidance
        self.class_emb = nn.Embedding(num_classes + 1, emb)

        self.stem = nn.Conv2d(in_ch, ch, 3, padding=1)

        self.d1 = ResBlock(ch,     ch,     emb)          # 32
        self.d2 = ResBlock(ch,     ch * 2, emb)          # 16
        self.d3 = ResBlock(ch * 2, ch * 2, emb)          # 8
        self.down = nn.AvgPool2d(2)

        self.m1 = ResBlock(ch * 2, ch * 2, emb)
        self.attn = SelfAttention(ch * 2)
        self.m2 = ResBlock(ch * 2, ch * 2, emb)

        self.u3 = ResBlock(ch * 4, ch * 2, emb)
        self.u2 = ResBlock(ch * 4, ch,     emb)
        self.u1 = ResBlock(ch * 2, ch,     emb)

        self.out = nn.Sequential(nn.GroupNorm(8, ch), nn.SiLU(), nn.Conv2d(ch, in_ch, 3, padding=1))

    def forward(self, x, t, y=None):
        emb = self.time_mlp(t)
        if y is None:                                        # None -> unconditional token
            y = torch.full((x.size(0),), NUM_CLASSES, device=x.device, dtype=torch.long)
        emb = emb + self.class_emb(y)

        h0 = self.stem(x)
        h1 = self.d1(h0, emb)                                # 32x32
        h2 = self.d2(self.down(h1), emb)                     # 16x16
        h3 = self.d3(self.down(h2), emb)                     # 8x8

        m = self.m2(self.attn(self.m1(h3, emb)), emb)

        u = self.u3(torch.cat([m, h3], 1), emb)
        u = F.interpolate(u, scale_factor=2, mode="nearest")
        u = self.u2(torch.cat([u, h2], 1), emb)
        u = F.interpolate(u, scale_factor=2, mode="nearest")
        u = self.u1(torch.cat([u, h1], 1), emb)
        return self.out(u)


model = DiffusionUNet().to(device)
print("params (M): %.2f" % (sum(p.numel() for p in model.parameters()) / 1e6))
with torch.no_grad():
    print("output:", tuple(model(torch.randn(2, 1, 32, 32, device=device),
                                torch.tensor([10, 300], device=device)).shape))

---
# 4. Training: it is just mean-squared error

This is the part that surprises people. The full DDPM training objective, after all the
variational-bound algebra in the paper, simplifies to:

$$L = \mathbb{E}_{x_0, t, \epsilon}\left[\; \lVert \epsilon - \epsilon_\theta(x_t, t) \rVert^2 \;\right]$$

Per step:

1. Take a real image x₀.
2. Sample a random timestep t and a random noise ε.
3. Build x_t in one line with `q_sample`.
4. Ask the network to predict ε. Take MSE.

No adversary, no balancing act, no discriminator to keep in check. The loss goes down and
the samples get better. That stability is the entire reason diffusion displaced GANs.

**Conditioning dropout.** We randomly replace the class label with the "unconditional"
token 10% of the time. That single line trains one network to be *both* a conditional and
an unconditional model, which Section 6 then exploits.

In [ ]:
def train_diffusion(model, epochs=12, lr=2e-4, p_uncond=0.1):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                                                steps_per_epoch=len(train_loader), pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    losses = []
    t0 = time.time()

    for ep in range(epochs):
        model.train(); running = 0.0
        for x0, y in train_loader:
            x0, y = x0.to(device, non_blocking=True), y.to(device, non_blocking=True)
            b = x0.size(0)

            t = torch.randint(0, T_STEPS, (b,), device=device)      # random timestep per image
            noise = torch.randn_like(x0)
            xt = q_sample(x0, t, noise)

            # classifier-free guidance dropout
            drop = torch.rand(b, device=device) < p_uncond
            y_in = torch.where(drop, torch.full_like(y, NUM_CLASSES), y)

            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=(device == "cuda")):
                loss = F.mse_loss(model(xt, t, y_in), noise)        # <-- the whole objective
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item()

        losses.append(running / len(train_loader))
        print("  epoch %2d/%d  loss %.4f  (%.0fs elapsed)" % (ep + 1, epochs, losses[-1], time.time() - t0))
    return losses


losses = train_diffusion(model, epochs=12)
plt.figure(figsize=(6, 3.5)); plt.plot(losses, marker="o")
plt.xlabel("epoch"); plt.ylabel("MSE"); plt.title("Diffusion training loss")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# Save a checkpoint - Colab disconnects, and you do not want to retrain.
torch.save(model.state_dict(), "diffusion_fashionmnist.pt")
print("saved diffusion_fashionmnist.pt")

---
# 5. Sampling

## 5.1 DDPM: the ancestral sampler

Start from pure noise x_T and walk back one step at a time. At each step, the model's
noise prediction gives the mean of the reverse distribution, and we add a little fresh
noise back:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t, t)\right) + \sigma_t z$$

Faithful, but it needs all T steps — a forward pass each. That is the price.

In [ ]:
@torch.no_grad()
def ddpm_sample(model, n=16, y=None, keep_trajectory=False):
    model.eval()
    x = torch.randn(n, CHANNELS, IMG_SIZE, IMG_SIZE, device=device)
    traj = []
    for i in reversed(range(T_STEPS)):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eps = model(x, t, y).float()
        mean = (x - (betas[i] / sqrt_one_minus_abar[i]) * eps) / alphas[i].sqrt()
        x = mean if i == 0 else mean + posterior_var[i].sqrt() * torch.randn_like(x)
        if keep_trajectory and i % (T_STEPS // 8) == 0:
            traj.append(x.clone().cpu())
    return x.clamp(-1, 1).cpu(), traj


def show_grid(imgs, title="", nrow=8):
    g = torchvision.utils.make_grid((imgs + 1) / 2, nrow=nrow, padding=2)
    plt.figure(figsize=(nrow * 1.3, 1.4 * (len(imgs) // nrow + 1)))
    plt.imshow(g.permute(1, 2, 0).numpy(), cmap="gray"); plt.axis("off")
    plt.title(title); plt.show()


t0 = time.time()
samples, traj = ddpm_sample(model, n=16, keep_trajectory=True)
print("DDPM: %d steps in %.1fs" % (T_STEPS, time.time() - t0))
show_grid(samples, "DDPM samples (%d steps, unconditional)" % T_STEPS)

In [ ]:
fig, ax = plt.subplots(len(traj), 1, figsize=(11, 1.5 * len(traj)))
for a, step in zip(ax, traj):
    g = torchvision.utils.make_grid((step[:8].clamp(-1, 1) + 1) / 2, nrow=8, padding=2)
    a.imshow(g.permute(1, 2, 0).numpy(), cmap="gray"); a.axis("off")
plt.suptitle("Reverse process: noise -> image (top = early, bottom = final)")
plt.tight_layout(); plt.show()

## 5.2 DDIM: the same model, 20x fewer steps

[DDIM](https://arxiv.org/abs/2010.02502) makes one observation: the training objective
never actually required the reverse process to be Markovian. You can construct a
**non-Markovian** reverse process with the same marginals, which lets you skip timesteps
— and if you set the injected noise to zero, sampling becomes **deterministic**.

At each step, first recover the model's estimate of the clean image,

$$\hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\epsilon_\theta}{\sqrt{\bar\alpha_t}}$$

then re-noise it to the *next* timestep on your (possibly sparse) schedule.

Same weights. No retraining. 400 steps → 50, at very little cost in quality. This is why
every production image model samples with 20–50 steps rather than 1000.

In [ ]:
@torch.no_grad()
def ddim_sample(model, n=16, steps=50, eta=0.0, y=None, guidance=0.0):
    model.eval()
    ts = torch.linspace(T_STEPS - 1, 0, steps).long().to(device)
    x = torch.randn(n, CHANNELS, IMG_SIZE, IMG_SIZE, device=device)

    for i in range(steps):
        t = torch.full((n,), ts[i], device=device, dtype=torch.long)

        if guidance > 0 and y is not None:
            # classifier-free guidance: one conditional and one unconditional prediction
            eps_c = model(x, t, y).float()
            eps_u = model(x, t, None).float()
            eps = eps_u + guidance * (eps_c - eps_u)
        else:
            eps = model(x, t, y).float()

        a_t = abar[ts[i]]
        a_prev = abar[ts[i + 1]] if i + 1 < steps else torch.tensor(1.0, device=device)

        x0_pred = ((x - (1 - a_t).sqrt() * eps) / a_t.sqrt()).clamp(-1, 1)
        sigma = eta * ((1 - a_prev) / (1 - a_t)).sqrt() * (1 - a_t / a_prev).sqrt()
        dir_xt = (1 - a_prev - sigma ** 2).clamp(min=0).sqrt() * eps
        x = a_prev.sqrt() * x0_pred + dir_xt
        if eta > 0 and i + 1 < steps:
            x = x + sigma * torch.randn_like(x)

    return x.clamp(-1, 1).cpu()


for steps in [10, 25, 50, 100]:
    t0 = time.time()
    s = ddim_sample(model, n=8, steps=steps)
    print("DDIM %3d steps : %.1fs" % (steps, time.time() - t0))
    show_grid(s, "DDIM, %d steps" % steps, nrow=8)

---
# 6. Classifier-free guidance: steering the model

We want to generate *a specific class*, and we want the samples to be unambiguous
examples of it. Feeding the class label alone gives conditional samples, but often weak
ones.

[Classifier-free guidance](https://arxiv.org/abs/2207.12598) sharpens them. Because we
trained with label dropout, the same network gives us both a conditional prediction
ε(x, t, y) and an unconditional one ε(x, t, ∅). Extrapolate away from the unconditional:

$$\tilde\epsilon = \epsilon(x,t,\varnothing) + w\,\big(\epsilon(x,t,y) - \epsilon(x,t,\varnothing)\big)$$

The difference is "the direction that makes this more like class y". Scaling it by w > 1
overshoots deliberately.

**This is the guidance scale you set in every text-to-image tool.** Same equation, with a
text embedding in place of the class label. It costs 2x compute per step, since you run
the network twice.

In [ ]:
y = torch.arange(NUM_CLASSES, device=device).repeat_interleave(4)   # 4 samples per class

for w in [0.0, 1.5, 3.0, 6.0]:
    s = ddim_sample(model, n=len(y), steps=50, y=y, guidance=w)
    show_grid(s, "classifier-free guidance w = %.1f  (rows = classes)" % w, nrow=4)

print("classes, in row order:")
for i, c in enumerate(LABELS):
    print("  %d %s" % (i, c))

Walk up the guidance scale and you should see the classic trade-off:

- **w = 0** — unconditional-ish. Diverse, but the class is often unclear.
- **w = 1.5–3** — the usual sweet spot. Clearly the right class, still varied.
- **w = 6+** — over-saturated, high-contrast, and noticeably *less diverse*. Guidance buys
  fidelity by spending diversity.

That trade-off is universal across diffusion models, and it is why every image generator
exposes this one number to the user.

In [ ]:
# Interpolating in the noise space: diffusion has a smooth latent structure too.
@torch.no_grad()
def interpolate(model, y_a, y_b, steps=8, guidance=3.0):
    torch.manual_seed(7)
    zs = torch.randn(1, CHANNELS, IMG_SIZE, IMG_SIZE, device=device)
    outs = []
    for lam in torch.linspace(0, 1, steps):
        # spherical interpolation between two class embeddings via guidance mixing
        ya = torch.full((1,), y_a, device=device, dtype=torch.long)
        yb = torch.full((1,), y_b, device=device, dtype=torch.long)
        x = zs.clone()
        ts = torch.linspace(T_STEPS - 1, 0, 50).long().to(device)
        for i in range(50):
            t = torch.full((1,), ts[i], device=device, dtype=torch.long)
            eps_u = model(x, t, None).float()
            eps = eps_u + guidance * ((1 - lam) * (model(x, t, ya).float() - eps_u)
                                      + lam * (model(x, t, yb).float() - eps_u))
            a_t = abar[ts[i]]
            a_prev = abar[ts[i + 1]] if i + 1 < 50 else torch.tensor(1.0, device=device)
            x0p = ((x - (1 - a_t).sqrt() * eps) / a_t.sqrt()).clamp(-1, 1)
            x = a_prev.sqrt() * x0p + (1 - a_prev).clamp(min=0).sqrt() * eps
        outs.append(x[0].clamp(-1, 1).cpu())
    return torch.stack(outs)


imgs = interpolate(model, 0, 9)     # T-shirt -> Ankle boot
show_grid(imgs, "conditioning interpolation: %s -> %s" % (LABELS[0], LABELS[9]), nrow=8)

---
# 7. Where this scales to

What you just built is the real thing, at small scale. Three changes turn it into a
production text-to-image model, and you will use all three in Lab 10:

| Your model | Stable Diffusion |
|---|---|
| Denoise in **pixel** space, 32x32 | Denoise in a VAE **latent** space, 64x64 latents for 512x512 images — ~48x less compute |
| Condition on a **class index** | Condition on **CLIP text embeddings** via cross-attention |
| U-Net backbone | U-Net (SD 1.5/2) or a **Transformer** backbone (DiT, SD3, and most models since) |

Two further developments worth knowing by name:

- **Latent Diffusion** ([Rombach et al.](https://arxiv.org/abs/2112.10752)) — the paper
  that made this affordable, and the direct basis of Stable Diffusion.
- **Flow matching / rectified flow** ([Lipman et al.](https://arxiv.org/abs/2210.02747),
  [SD3](https://arxiv.org/abs/2403.03206)) — reframes the process as learning a
  straight-line velocity field rather than a noise-removal chain. Straighter paths mean
  fewer sampling steps. This is where most new models are.

---
# Exercises

**1. Schedule matters.** Swap the cosine schedule for the original linear one
(`torch.linspace(1e-4, 0.02, T)`). Plot both ᾱ curves together, retrain, and compare
samples. Explain why the linear schedule wastes capacity on 32x32 images. (Hint: look at
how early ᾱ approaches zero.)

**2. Predict x₀ instead of ε.** Change the target from `noise` to `x0` and adjust the
samplers accordingly. Train both and compare. Which is more stable early in training, and
which at large t? This is a real design axis — some models predict **v**, a blend of the
two.

**3. Guidance sweep, measured.** Generate 200 samples per class at w = 0, 1, 2, 4, 8.
Train a small classifier on FashionMNIST and use it to measure (a) how often the sample is
recognised as the requested class, and (b) the diversity of the samples (e.g. mean
pairwise distance). Plot both against w and locate the sweet spot quantitatively.

**4. Step-count vs. quality.** Sample with DDIM at 5, 10, 20, 50, 100, 200 steps from the
same starting noise. Show the grids side by side. Where does quality saturate? Then set
`eta=1.0` (stochastic DDIM) and repeat — how does the answer change?

**5. DDIM is deterministic — prove it.** With `eta=0`, run the same starting noise twice
and confirm the outputs are identical. Then interpolate between two *starting noise
vectors* (spherical interpolation) and show the output morphs smoothly. Why does this
fail with `eta=1.0`?

**6. Inpainting for free.** Implement masked sampling: at every reverse step, replace the
known region of x_t with `q_sample(x_known, t)` and let the model fill only the unknown
region. Mask out the bottom half of a test image and complete it. No retraining required
— explain why this works.

**7. Scale up.** Retrain on CIFAR-10 (3 channels, 32x32) with `ch=96` and 30 epochs.
Report training time and show samples. This is roughly the setting of the original DDPM
paper's CIFAR experiments, at a fraction of the compute.